# 03 — Train a neural network in Rust

**AI notebook:** a tiny **two-layer neural network** learns the nonlinear XOR problem using **backpropagation**. Everything is implemented in Rust using the standard library; no PyTorch, Python ML code, pretrained models, cloud APIs or downloads.

This is an educational example of neural-network training, not a large language model. Run all code cells **in order** with the Rust kernel.

## 1. Synthetic training data

XOR outputs 1 when exactly one input is 1. A single linear classifier cannot solve XOR, so we use hidden neurons with nonlinear activations.

In [1]:
let samples: [([f64; 2], f64); 4] = [
    ([0.0, 0.0], 0.0),
    ([0.0, 1.0], 1.0),
    ([1.0, 0.0], 1.0),
    ([1.0, 1.0], 0.0),
];
println!("Synthetic XOR samples: {}", samples.len());

Synthetic XOR samples: 4


## 2. Define the model

Architecture: **2 inputs → 6 tanh hidden neurons → 1 sigmoid output**.

We use deterministic, non-identical starting weights so results are reproducible. Training uses binary cross-entropy's output gradient `prediction - target`.

In [2]:
const HIDDEN: usize = 6;
struct TinyNet {
    weights: [[f64; 2]; HIDDEN],
    biases: [f64; HIDDEN],
    output_weights: [f64; HIDDEN],
    output_bias: f64,
}
fn sigmoid(z: f64) -> f64 { 1.0 / (1.0 + (-z).exp()) }
impl TinyNet {
    fn new() -> Self {
        let mut net = Self {
            weights: [[0.0; 2]; HIDDEN],
            biases: [0.0; HIDDEN],
            output_weights: [0.0; HIDDEN],
            output_bias: 0.0,
        };
        for h in 0..HIDDEN {
            let k = h as f64;
            net.weights[h] = [(k * 1.7).sin() * 0.8, (k * 2.3 + 0.8).cos() * 0.8];
            net.output_weights[h] = (k * 0.9 + 0.3).sin() * 0.5;
        }
        net
    }
    fn hidden(&self, x: [f64; 2]) -> [f64; HIDDEN] {
        let mut a = [0.0; HIDDEN];
        for h in 0..HIDDEN {
            a[h] = (self.weights[h][0] * x[0]
                + self.weights[h][1] * x[1] + self.biases[h]).tanh();
        }
        a
    }
    fn predict(&self, x: [f64; 2]) -> f64 {
        let a = self.hidden(x);
        let mut sum = self.output_bias;
        for h in 0..HIDDEN { sum += self.output_weights[h] * a[h]; }
        sigmoid(sum)
    }
    fn train_step(&mut self, x: [f64; 2], target: f64, rate: f64) {
        let a = self.hidden(x);
        let prediction = self.predict(x);
        let output_gradient = prediction - target;
        let mut hidden_gradients = [0.0; HIDDEN];
        // Compute hidden gradients before updating the output weights.
        for h in 0..HIDDEN {
            hidden_gradients[h] = output_gradient * self.output_weights[h]
                * (1.0 - a[h] * a[h]);
        }
        for h in 0..HIDDEN {
            self.output_weights[h] -= rate * output_gradient * a[h];
            for j in 0..2 {
                self.weights[h][j] -= rate * hidden_gradients[h] * x[j];
            }
            self.biases[h] -= rate * hidden_gradients[h];
        }
        self.output_bias -= rate * output_gradient;
    }
}

## 3. Train with backpropagation

Every epoch visits all four synthetic examples and updates the weights. Unlike random forest, this model learns by gradient descent.

In [3]:
let mut model = TinyNet::new();
let learning_rate = 0.1;
for _epoch in 0..8000 {
    for &(input, expected) in &samples {
        model.train_step(input, expected, learning_rate);
    }
}
println!("Finished training the neural network.");

Finished training the neural network.


## 4. Test predictions

The output is a probability from 0 to 1. Values above 0.5 predict class 1; values below 0.5 predict class 0.

In [4]:
let mut correct = 0;
for &(input, expected) in &samples {
    let probability = model.predict(input);
    let predicted = (probability >= 0.5) as u8;
    if predicted as f64 == expected { correct += 1; }
    println!("{:?} -> P(XOR=1)={:.4} | class={} | expected={}",
        input, probability, predicted, expected as u8);
}
println!("Training accuracy: {}/{}", correct, samples.len());
assert_eq!(correct, 4);

[0.0, 0.0] -> P(XOR=1)=0.0001 | class=0 | expected=0


[0.0, 1.0] -> P(XOR=1)=0.9995 | class=1 | expected=1


[1.0, 0.0] -> P(XOR=1)=0.9992 | class=1 | expected=1


[1.0, 1.0] -> P(XOR=1)=0.0009 | class=0 | expected=0


Training accuracy: 4/4


## Experiment

Try reducing epochs (8000 → 200) or the hidden-layer width (6 → 2), then restart the kernel and run the notebook again. Watch how optimization and model capacity affect predictions.

For a different approach to nonlinear classification, go back to [02 — Random Forest](02_random_forest.ipynb).

**Note:** Because XOR has only four unique examples, the reported metric is **training accuracy**. It is not evidence that a model generalizes to unseen real-world data.